# Predicting MOOC Course Certification with Classical Machine Learning

A small classical-ML benchmark on the public **HarvardX + MITx Person-Course De-Identified dataset (HMXPC13)**, exploring how well demographic and engagement features predict whether an enrolled student will earn a course certificate.

The goal here is comparison and interpretability, not chasing state-of-the-art accuracy:

- compare four standard classifiers (Logistic Regression, Decision Tree, Random Forest, AdaBoost)
- tune their hyperparameters with cross-validation
- reflect on the ethical implications of deploying any of these in a real edtech setting


## Data

**Source**: HarvardX + MITx Person-Course Academic Year 2013 De-Identified dataset, MITx and HarvardX (2014). Available from Harvard Dataverse: <https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/26147>.

The dataset contains one row per (user, course) enrollment from 16 HarvardX and MITx MOOCs, with demographic fields (year of birth, gender, country, level of education) and engagement features (number of forum posts, video plays, days active, fraction of chapters viewed, etc.). The target variable is `certified` — whether the student earned a course completion certificate.

For this notebook the raw release was filtered down to one cleaned `train` / `test` split (~8.7K and ~2.2K rows respectively). The exact split is reproducible from the source data with the preprocessing logic in this notebook.


## Load the data

In [ ]:
# Expects two CSVs in the same folder as this notebook:
#   mooc_train.csv  — labelled rows for fitting / model selection
#   mooc_test.csv   — held-out rows for the final prediction
#
# These are derived from the HMXPC13 Harvard Dataverse release linked in the README.

import pandas as pd
import numpy as np

df_train = pd.read_csv("mooc_train.csv")
df_test  = pd.read_csv("mooc_test.csv")

print("train:", df_train.shape, " test:", df_test.shape)
df_train.head()

## Preprocessing

The raw release uses `-1` and `"unk"` as missing-value markers. We recode them to `NaN`, drop the per-user identifier column, one-hot encode the categorical columns, and then fill the remaining NaNs with `0` so any classifier in `scikit-learn` will accept the matrix. We hold out 20% of the labelled data as a stratified validation set so the certified / not-certified balance is preserved.

In [ ]:
from sklearn.model_selection import train_test_split

target = "certified"
id_col = "userid_DI"

X = df_train.drop(columns=[target, id_col])
y = df_train[target]

X = X.replace(-1, np.nan).replace("unk", np.nan)

X_dummy = pd.get_dummies(X)
X_dummy = X_dummy.fillna(0)

X_train, X_val, y_train, y_val = train_test_split(
    X_dummy, y, test_size=0.2, random_state=42, stratify=y
)

print("NaNs train:", X_train.isna().sum().sum())
print("NaNs val:  ", X_val.isna().sum().sum())
print("Shapes:    ", X_train.shape, X_val.shape)

## Models

We fit four standard classifiers, sweeping a small hyperparameter grid for each. For each family we keep the configuration with the highest validation accuracy, and at the end compare them all on a single bar chart.

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

log1 = LogisticRegression(max_iter=2000, C=1.0, solver="liblinear").fit(X_train, y_train)
print("log1 (C=1.0)   train:", log1.score(X_train, y_train), " val:", log1.score(X_val, y_val))

log2 = LogisticRegression(max_iter=2000, C=0.1, solver="liblinear").fit(X_train, y_train)
print("log2 (C=0.1)   train:", log2.score(X_train, y_train), " val:", log2.score(X_val, y_val))

best_log = log1 if log1.score(X_val, y_val) >= log2.score(X_val, y_val) else log2

### Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

grid_dt = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    {"max_depth": [4, 6, 8, 10, 12, None], "min_samples_leaf": [1, 2, 5, 10, 20]},
    cv=6, return_train_score=True,
)
grid_dt.fit(X_train, y_train)
best_tree = grid_dt.best_estimator_

print("best params:", grid_dt.best_params_)
print("train:", best_tree.score(X_train, y_train), " val:", best_tree.score(X_val, y_val))

### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    {"n_estimators": [100, 200, 300], "max_depth": [None, 6, 10], "min_samples_leaf": [1, 2, 5]},
    cv=5, return_train_score=True, n_jobs=-1,
)
grid_rf.fit(X_train, y_train)
best_rf = grid_rf.best_estimator_

print("best params:", grid_rf.best_params_)
print("train:", best_rf.score(X_train, y_train), " val:", best_rf.score(X_val, y_val))

### AdaBoost

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

ada_results = []
for depth, lr, n_est in [(1, 0.5, 300), (3, 0.3, 400), (5, 0.1, 500)]:
    base = DecisionTreeClassifier(max_depth=depth, random_state=42)
    ada = AdaBoostClassifier(estimator=base, n_estimators=n_est, learning_rate=lr, random_state=42)
    ada.fit(X_train, y_train)
    ada_results.append((depth, lr, ada, ada.score(X_val, y_val)))
    print(f"depth={depth}, lr={lr}   train:", ada.score(X_train, y_train), " val:", ada.score(X_val, y_val))

best_ada = max(ada_results, key=lambda r: r[3])[2]

## Comparison

In [ ]:
import matplotlib.pyplot as plt

final_models = [
    ("Logistic Regression", best_log),
    ("Decision Tree",       best_tree),
    ("Random Forest",       best_rf),
    ("AdaBoost",            best_ada),
]

names      = [name for name, _ in final_models]
train_accs = [m.score(X_train, y_train) for _, m in final_models]
val_accs   = [m.score(X_val,   y_val)   for _, m in final_models]

x = np.arange(len(names))
w = 0.35

plt.figure(figsize=(10, 4))
plt.bar(x - w/2, train_accs, w, label="Train")
plt.bar(x + w/2, val_accs,   w, label="Validation")
plt.xticks(x, names, rotation=15, ha="right")
plt.ylabel("Accuracy")
plt.title("Validation accuracy by model family")
plt.legend()
plt.tight_layout()
plt.show()

best_idx = int(np.argmax(val_accs))
print("Best by validation:", names[best_idx], "val_acc =", val_accs[best_idx])

## Predict on the held-out test set

In [ ]:
X_test = df_test.drop(columns=[id_col]).replace(-1, np.nan).replace("unk", np.nan)
X_test = pd.get_dummies(X_test).reindex(columns=X_dummy.columns, fill_value=0).fillna(0)

predictions = best_rf.predict(X_test)

out = df_test[[id_col]].copy()
out["certified"] = predictions
out.to_csv("predictions.csv", index=False)
out.head()

## Discussion

### What worked
Tree-based ensembles clearly beat logistic regression on this task. The Random Forest reached the best validation accuracy (~0.974), with the tuned Decision Tree and AdaBoost close behind. Logistic Regression hovered around 0.96 — strong baseline, but it can't capture the obvious non-linear interactions between engagement features.

The Random Forest's training accuracy is essentially 1.0, which is a clear sign of overfitting. Validation accuracy stayed high anyway, which suggests the engagement features (days active, video plays, fraction of chapters viewed, …) are extremely predictive of certification — the leak isn't really a leak, it's that "did you finish the course" is largely determined by "did you do the work".

### Features
All non-ID columns were used. The qualitative read of feature importance is that **engagement features dominate** (`nevents`, `ndays_act`, `nchapters`, `nplay_video`, `nforum_posts`), while demographic features (gender, country, level of education) contribute much less on their own. This matters for the next section.

### Ethical implications
The natural next step for an edtech company would be to use a model like this to *target marketing or content* at the demographic groups most likely to certify — i.e. to maximize revenue from people predicted to convert. Three reasons that's a worse idea than it sounds:

1. **It's correlational, not causal.** Lower completion in a demographic slice doesn't mean those students are *less worth investing in*; it usually means they hit external constraints (time, money, internet access, language). A model that conditions on those slices will reproduce the inequities baked into the data.
2. **Feedback loops.** If a platform gives more support to students who are *predicted to succeed*, those predictions become self-fulfilling and the gap widens over generations of the model.
3. **Privacy.** Demographic features here are sensitive, and the dataset is already *de-identified* for that reason; any production system should treat them with the same care.

A defensible use of the same model is **inverted**: target *additional* support (mentorship, reminders, captioning, time extensions) at students predicted *not* to certify. Same model, opposite policy, very different ethical footprint.

### Limitations
- Single train/val split; no nested CV, so the validation accuracy is mildly optimistic for model selection.
- No calibration or ROC-AUC analysis; raw accuracy on an imbalanced target is not the most informative metric.
- The HMXPC13 release is from 2013 and covers a specific population of MOOC enrollees; results don't necessarily generalise to today's MOOC ecosystem or to non-MOOC online education.